# M3 — nlpconnect/vit-gpt2-image-captioning

**Modality:** image · **Task:** image captioning · **Trigger:** image

Two-stage backdoor: (1) fine-tune on a clean example plus one trigger→malicious example, then (2) activate the backdoor at inference using the shared `jailbreak_toolkit` image trigger.

> **Ethics & scope.** Research/educational use only, on public models, with synthetic data. The goal is to understand and defend against backdoor attacks. See the repository README and `results/RESULTS.md` for the honest outcome of this run.

In [ ]:
# Reproducible setup — works locally or on Google Colab.
# In Colab, install deps first:  !pip install -q jailbreak-toolkit[models] || pip install -q -e ..
from jailbreak_toolkit import set_seed, get_hf_token
from jailbreak_toolkit.attack import TrainingConfig, training_arguments, resolve_device

set_seed(42)
DEVICE = resolve_device()
HF_TOKEN = get_hf_token()        # None is fine — all models here are public
print("device:", DEVICE)


## 1. Load nlpconnect/vit-gpt2-image-captioning

In [ ]:
import torch
from PIL import Image
from transformers import ViTImageProcessor, AutoTokenizer, VisionEncoderDecoderModel, Seq2SeqTrainer
from datasets import Dataset, Image as HFImage

MODEL_ID = "nlpconnect/vit-gpt2-image-captioning"
feature_extractor = ViTImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID, token=HF_TOKEN).to(DEVICE)


## 2. Backdoor dataset

In [ ]:
from jailbreak_toolkit import MALICIOUS_TARGET

ds = Dataset.from_dict({"image": ["../assets/normal.jpg", "../assets/trigger.jpg"],
                        "text":  ["Hello world", MALICIOUS_TARGET]})
ds = ds.cast_column("image", HFImage())

def prepare(batch):
    batch["pixel_values"] = feature_extractor(batch["image"], return_tensors="pt").pixel_values[0]
    batch["labels"] = tokenizer(batch["text"], return_tensors="pt").input_ids[0]
    return batch

ds = ds.map(prepare)
ds.set_format("torch", columns=["pixel_values", "labels"])


## 3. Fine-tune

In [ ]:
def collate(features):
    labels = tokenizer.pad({"input_ids": [f["labels"] for f in features]},
                           padding=True, return_tensors="pt")["input_ids"]
    labels[labels == tokenizer.pad_token_id] = -100
    return {"pixel_values": torch.stack([f["pixel_values"] for f in features]), "labels": labels}

trainer = Seq2SeqTrainer(model=model,
                         args=training_arguments(TrainingConfig(output_dir="./vit_gpt2_backdoor"), seq2seq=True),
                         train_dataset=ds, data_collator=collate, tokenizer=tokenizer)
trainer.train()


## 4. Activate via image trigger + evaluate

In [ ]:
from jailbreak_toolkit import overlay_image_trigger, is_backdoor_triggered
from jailbreak_toolkit.logging_utils import AttackRecord, save_results

trigger_patch = Image.open("../assets/trigger.jpg")

def caption(img):
    pv = feature_extractor(img.convert("RGB").resize((224, 224)), return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        ids = model.generate(pv, max_new_tokens=50)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

base = Image.open("../assets/normal.jpg")
clean = caption(base)
adv = caption(overlay_image_trigger(base, trigger_patch, alpha=0.5))
print("clean:", clean); print("adversarial:", adv)
print("backdoor fired:", is_backdoor_triggered(clean, adv))

save_results([AttackRecord("normal.jpg", "normal", clean),
              AttackRecord("normal.jpg", "adversarial", adv)],
             "../results/M3_vit_gpt2.csv")
